In [1]:
import pandas as pd

# 1. Membaca file dataset
df = pd.read_csv('sales_transactions_raw.csv')

# 2. Menampilkan 5 baris pertama data
print("--- 5 Baris Pertama Data ---")
display(df.head())

# 3. Menampilkan informasi ringkas (tipe data dan jumlah baris/kolom)
print("\n--- Informasi Struktur Data ---")
df.info()

# 4. Mengecek apakah ada data yang kosong (null/missing values)
print("\n--- Jumlah Data yang Kosong per Kolom ---")
print(df.isnull().sum())

--- 5 Baris Pertama Data ---


,order_id,order_date,region,channel,customer_id,customer_type,sales_rep,category,product,qty,discount,status,unit_price,revenue,cost
0,ORD-100001,2024-09-01,Jakarta,Offline,C00438,UMKM,SR02,Smartphone,Smartphone Lumo X,4,0.1,Completed,5500000.0,19800000.0,19274000.0
1,ORD-100002,2024-09-01,Jawa Timur,Offline,K0002,Korporat,SR09,Peralatan Kantor,Kursi Ergonomis E1,20,0.1,Completed,1900000.0,34200000.0,26338000.0
2,ORD-100003,2024-09-01,Jawa Barat,Online,C01918,Retail,SYS-ONLINE,Laptop,Laptop Axio Pro 15,1,0.1,Completed,13000000.0,11700000.0,11642500.0
3,ORD-100004,2024-09-01,Kalimantan,Offline,C00108,Retail,SR14,Aksesoris,Keyboard Mekanik K7,1,0.0,Completed,650000.0,650000.0,401000.0
4,ORD-100005,2024-09-01,Sulawesi,Online,C01567,Retail,SYS-ONLINE,Smartphone,Smartphone Lumo X,2,0.1,Completed,5500000.0,9900000.0,9743000.0



--- Informasi Struktur Data ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7309 entries, 0 to 7308
Data columns (total 15 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   order_id       7309 non-null   object 
 1   order_date     7309 non-null   object 
 2   region         7175 non-null   object 
 3   channel        7309 non-null   object 
 4   customer_id    7309 non-null   object 
 5   customer_type  7309 non-null   object 
 6   sales_rep      7309 non-null   object 
 7   category       7309 non-null   object 
 8   product        7309 non-null   object 
 9   qty            7309 non-null   int64  
 10  discount       7309 non-null   float64
 11  status         7309 non-null   object 
 12  unit_price     7309 non-null   float64
 13  revenue        7309 non-null   float64
 14  cost           6789 non-null   float64
dtypes: float64(4), int64(1), object(10)
memory usage: 856.7+ KB

--- Jumlah Data yang Kosong per Kolom ---
orde

In [3]:
# --- PERBAIKAN TAHAP CLEANING & PROCESSING ---
df_clean = df.copy()

# 1. Menangani baris dengan tanggal kosong
df_clean = df_clean.dropna(subset=['order_date'])

# 2. Mengisi nilai kosong pada diskon dengan 0
df_clean['discount'] = df_clean['discount'].fillna(0)

# 3. Menangani cost yang kosong
df_clean = df_clean.dropna(subset=['cost'])

# 4. Memastikan format tanggal (menggunakan errors='coerce' agar aman dari error teks)
df_clean['order_date'] = pd.to_datetime(df_clean['order_date'], errors='coerce')

# Membuat kolom bagian tanggal
df_clean['year'] = df_clean['order_date'].dt.year
df_clean['month'] = df_clean['order_date'].dt.month
df_clean['quarter'] = df_clean['order_date'].dt.quarter
df_clean['year_month'] = df_clean['order_date'].dt.to_period('M').astype(str)

# Menghitung Net Revenue & Gross Profit
def calculate_financials(row):
    if row['status'] == 'Completed':
        net_rev = row['revenue']
        gross_prof = row['revenue'] - (row['cost'] * row['qty'])
    else:
        net_rev = 0
        gross_prof = 0
    return pd.Series([net_rev, gross_prof])

df_clean[['net_revenue', 'gross_profit']] = df_clean.apply(calculate_financials, axis=1)

# Menghitung Gross Margin (%)
df_clean['gross_margin_pct'] = (df_clean['gross_profit'] / df_clean['net_revenue']) * 100
df_clean['gross_margin_pct'] = df_clean['gross_margin_pct'].fillna(0)

# Menampilkan ringkasan data bersih
print("--- Berhasil! Data Bersih & Diproses ---")
display(df_clean[['order_id', 'order_date', 'status', 'net_revenue', 'gross_profit', 'gross_margin_pct']].head())

--- Berhasil! Data Bersih & Diproses ---


,order_id,order_date,status,net_revenue,gross_profit,gross_margin_pct
0,ORD-100001,2024-09-01,Completed,19800000.0,-57296000.0,-289.373737
1,ORD-100002,2024-09-01,Completed,34200000.0,-492560000.0,-1440.233918
2,ORD-100003,2024-09-01,Completed,11700000.0,57500.0,0.491453
3,ORD-100004,2024-09-01,Completed,650000.0,249000.0,38.307692
4,ORD-100005,2024-09-01,Completed,9900000.0,-9586000.0,-96.828283


In [4]:
# --- TAHAP 4: MEMBUAT TABEL RINGKASAN (SUMMARY TABLES) ---

# 1. Ringkasan Bulanan (Monthly Summary)
print("--- 1. Ringkasan Bulanan ---")
monthly_summary = df_clean.groupby('year_month').agg(
    total_net_revenue=('net_revenue', 'sum'),
    total_gross_profit=('gross_profit', 'sum')
).reset_index()
monthly_summary['gross_margin_pct'] = (monthly_summary['total_gross_profit'] / monthly_summary['total_net_revenue']) * 100
display(monthly_summary)

# 2. Ringkasan Berdasarkan Region x Category
print("\n--- 2. Ringkasan Berdasarkan Region & Kategori ---")
region_cat_summary = df_clean.groupby(['region', 'category']).agg(
    total_net_revenue=('net_revenue', 'sum'),
    total_gross_profit=('gross_profit', 'sum')
).reset_index()
region_cat_summary['gross_margin_pct'] = (region_cat_summary['total_gross_profit'] / region_cat_summary['total_net_revenue']) * 100
display(region_cat_summary.head(10))

# 3. Top 10 Pelanggan Berdasarkan Net Revenue
print("\n--- 3. Top 10 Pelanggan (Customer ID) Berdasarkan Net Revenue ---")
top_customers = df_clean.groupby('customer_id').agg(
    total_net_revenue=('net_revenue', 'sum'),
    total_orders=('order_id', 'count')
).reset_index().sort_values(by='total_net_revenue', ascending=False).head(10)
display(top_customers)

--- 1. Ringkasan Bulanan ---


,year_month,total_net_revenue,total_gross_profit,gross_margin_pct
0,2024-09,1.822998e+09,-1.438721e+10,-789.206059
1,2024-10,3.447258e+09,-3.094669e+11,-8977.188174
2,2024-11,2.212091e+09,-3.242864e+10,-1465.972300
3,2024-12,2.565300e+09,-3.281590e+10,-1279.222493
4,2025-01,1.586229e+09,-1.369577e+10,-863.416899
5,2025-02,1.925160e+09,-1.301064e+10,-675.821128
6,2025-03,2.752068e+09,-3.199403e+10,-1162.544939
7,2025-04,1.756988e+09,-9.693436e+09,-551.707564
8,2025-05,2.091810e+09,-1.004476e+10,-480.194731
9,2025-06,2.127088e+09,-1.531537e+10,-720.015782



--- 2. Ringkasan Berdasarkan Region & Kategori ---


,region,category,total_net_revenue,total_gross_profit,gross_margin_pct
0,DKI Jakarta,Aksesori,1950000.0,-1.678800e+06,-86.092308
1,DKI Jakarta,Aksesoris,22700000.0,-2.592802e+08,-1142.203524
2,DKI Jakarta,Laptop,213800000.0,-4.034460e+08,-188.702526
3,DKI Jakarta,Peralatan Kantor,58391050.0,-1.237913e+09,-2120.039730
4,DKI Jakarta,Printer & Tinta,12439000.0,-2.417780e+07,-194.370930
5,DKI Jakarta,Printer&Tinta,0.0,0.000000e+00,NaN
6,DKI Jakarta,Smartphone,66897660.0,-6.541517e+07,-97.783936
7,DKI Jakarta,smartphone,8400000.0,-1.363290e+07,-162.296429
8,JKT,ATK,3610000.0,-1.644400e+06,-45.551247
9,JKT,Aksesoris,2963050.0,-1.707386e+06,-57.622581



--- 3. Top 10 Pelanggan (Customer ID) Berdasarkan Net Revenue ---


,customer_id,total_net_revenue,total_orders
1741,C01919,1.708595e+09,4
2150,C02374,5.080000e+08,2
2276,K0102,5.042520e+08,1
1587,C01747,5.033000e+08,3
668,C00737,5.002250e+08,2
2343,K0169,4.475440e+08,4
2301,K0127,4.076695e+08,6
2325,K0151,3.570365e+08,8
2205,K0031,3.350493e+08,8
2185,K0011,2.944346e+08,6
